# GPT-5.4-mini · Single Juror vs Committee

Symposia now validates each input as a **holistic single claim** by default, then scores it for **support**, **contradiction**, and **sufficiency**.

This notebook compares two modes on seven claims — all using the same model (`gpt-5.4-mini`) so the only variable is **single juror vs four diverse personas**:

| Mode | What happens |
|---|---|
| **Single juror** | One LLM call via `model="openai:gpt-5.4-mini"` — generic system prompt |
| **Committee of 4** | Four `gpt-5.4-mini` juror personas, each with a **personality-injected system prompt** derived from its `Profile` (stance, evidence demand, safety bias) |

Each juror's system prompt is built from its registered `Profile` — e.g. the Risk Sentinel receives *"Your reviewing stance is risk-first. Your evidence demand is high. Your safety bias is high."* while the Balanced Reviewer gets *"Your reviewing stance is balanced. Your evidence demand is medium."*

- This notebook uses the default holistic path (`decomposition_mode="holistic"`).

> **Prerequisites** — `pip install -e .` from repo root · `OPENAI_API_KEY` set · this notebook makes **live** API calls

In [1]:
from __future__ import annotations
import os, sys, statistics
from pathlib import Path
from pprint import pprint

import nest_asyncio
nest_asyncio.apply()

ROOT = Path.cwd()
if ROOT.name == "examples":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from symposia import validate
from symposia.env import load_env
from symposia.models.routing import JurorRoutingConfig

load_env()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is required.")

# Single juror baseline: one direct model call
SINGLE_MODEL = "openai:gpt-5.4-mini"

# System defined: Committee setup - same model, different reviewer personas
JUROR_PROFILES = [
    ("balanced_reviewer_v1",      600),
    ("sceptical_verifier_v1",     600),
    ("evidence_maximalist_v1",    700),
    ("risk_sentinel_v1",          700),
]

COMMITTEE_ROUTE = JurorRoutingConfig(
    version="v1",  # schema version for this routing object
    route_set_id="notebook_gpt54mini_committee",  # label for this route config
    stage="round0",  # first-pass review stage
    domain="all",  # allow this route for any domain
    output_schema="juror_decision_v1",  # expected juror response format
    guardrails={
        "max_premium_jurors_per_run": 0,  # disallow expensive premium jurors
        "require_provider_diversity": False,  # don't force multiple providers
        "require_model_family_diversity": False,  # don't force mixed model families
        "premium_allowed_in_round0": False,  # no premium models in first pass
    },
    assignments=[
        {
            "slot_id": f"round0_{pid.split('_')[0]}",  # unique juror slot name
            "profile_id": pid,  # reviewer persona to apply
            "provider": "openai",  # model vendor
            "model": "gpt-5.4-mini",  # exact model name
            "model_family": "gpt-5.4",  # model family grouping
            "tier": "small_capable",  # capability/cost tier label
            "timeout_seconds": 12,  # max wait per juror call
            "max_output_tokens": tokens,  # response length cap
            "fallback": {
                "provider": "openai",  # backup vendor if needed
                "model": "gpt-5.4-mini",  # backup model
                "model_family": "gpt-5.4",  # backup family
            },
        }
        for pid, tokens in JUROR_PROFILES
    ],
)

print(f"Single    : {SINGLE_MODEL}")
print(f"Committee : gpt-5.4-mini × {len(COMMITTEE_ROUTE.assignments)} jurors -> "
      f"{[a.profile_id for a in COMMITTEE_ROUTE.assignments]}")

Single    : openai:gpt-5.4-mini
Committee : gpt-5.4-mini × 4 jurors -> ['balanced_reviewer_v1', 'sceptical_verifier_v1', 'evidence_maximalist_v1', 'risk_sentinel_v1']


In [2]:
def summarize(result):
    scores = list(result.aggregated_by_subclaim.values())
    return {
        "raw_subclaims": len(getattr(result, "extracted_subclaims", [])),
        "aggregated_units": len(scores),
        "should_stop": result.completion.should_stop,
        "reason": result.completion.reason,
        "avg_support": round(statistics.mean(s.support_score for s in scores), 3),
        "avg_contradiction": round(statistics.mean(s.contradiction_score for s in scores), 3),
        "avg_sufficiency": round(statistics.mean(s.sufficiency_score for s in scores), 3),
        "low_sufficiency": sum(1 for s in scores if s.sufficiency_score < 0.70),
        "high_contradiction": sum(1 for s in scores if s.contradiction_score >= 0.35),
    }

def run_case(name, domain, content):
    """Run single juror and committee side-by-side, return comparison dict."""
    single_raw    = validate(content=content, domain=domain, model=SINGLE_MODEL, live=True)
    committee_raw = validate(content=content, domain=domain, routing=COMMITTEE_ROUTE, live=True)
    return {
        "case": name,
        "single": summarize(single_raw),
        "committee": summarize(committee_raw)
    }

In [3]:
USE_CASES = [
    {"name": "Forecast: rate cut within two quarters",
     "domain": "finance",
     "content": "Given cooling inflation, softer labor data, and recent central bank messaging, "
                "there is a high chance the central bank will cut rates within the next two quarters."},
    
    {"name": "AGI: capability claim without operational definition",
    "domain": "general",
    "content": "This model is effectively AGI because it can code, write essays, answer questions, and use tools across many domains."},
    
    
    {"name": "Control: diversification reduces risk",
     "domain": "finance",
     "content": "Diversifying a portfolio across uncorrelated asset classes reduces "
                "portfolio-specific risk without necessarily lowering expected returns."},
    
    {"name": "Mixed-truth: vitamin D overclaim",
     "domain": "medical",
     "content": "Vitamin D prevents all respiratory infections and has no downside for almost everyone."},
    {"name": "High-stakes: stop anticoagulant before dental procedure",
     "domain": "medical",
     "content": "Stop your anticoagulant medication today before your dental procedure. "
                "There is no need to contact your clinician first."},
    {"name": "Legal: liability transfers everywhere",
     "domain": "legal",
     "content": "This policy always transfers liability to the vendor in every jurisdiction, "
                "so it is safe to approve without local counsel review."},
    {"name": "AI risk: enterprise-safe by architecture alone",
     "domain": "general",
     "content": "Because the system uses retrieval and role prompts, it is enterprise-safe "
                "by design and does not need further governance review."},
]

for c in USE_CASES:
    print(f"  • {c['name']}  [{c['domain']}]")

  • Forecast: rate cut within two quarters  [finance]
  • AGI: capability claim without operational definition  [general]
  • Control: diversification reduces risk  [finance]
  • Mixed-truth: vitamin D overclaim  [medical]
  • High-stakes: stop anticoagulant before dental procedure  [medical]
  • Legal: liability transfers everywhere  [legal]
  • AI risk: enterprise-safe by architecture alone  [general]


## 1 — Forecast case (deep dive)

In the stored output below, the forecast case shows a limit case rather than a committee advantage: both the single juror and the committee mark the claim as an `escalation_candidate` with zero support, zero contradiction, and zero sufficiency. The useful takeaway is that both paths can refuse to overstate confidence when the claim is too under-evidenced to score cleanly.

In [4]:
forecast = run_case(**USE_CASES[0])
pprint(forecast)

{'case': 'Forecast: rate cut within two quarters',
 'committee': {'aggregated_units': 1,
               'avg_contradiction': 0.0,
               'avg_sufficiency': 0.0,
               'avg_support': 0.0,
               'high_contradiction': 0,
               'low_sufficiency': 1,
               'raw_subclaims': 0,
               'reason': 'escalation_candidate',
               'should_stop': False},
 'single': {'aggregated_units': 1,
            'avg_contradiction': 0.0,
            'avg_sufficiency': 0.0,
            'avg_support': 0.0,
            'high_contradiction': 0,
            'low_sufficiency': 1,
            'raw_subclaims': 0,
            'reason': 'escalation_candidate',
            'should_stop': False}}


## 2 — All seven cases

Look for three patterns in the stored results: exact convergence, softer committee calibration, and full agreement that a claim should escalate. The control case is the clean sanity check where both modes come out decisive and fully supported.

In [5]:
results = [run_case(**c) for c in USE_CASES]

### How to read the table below

Symposia still exposes these aggregates via `aggregated_by_subclaim`; in the holistic mode used here, that usually means one scored unit per claim.

| Metric | What it means | "Good" claim | "Bad" claim |
|---|---|---|---|
| `avg_support` | Mean evidence support across scored units | >= 0.70 | < 0.70 |
| `avg_contradiction` | Mean contradictory evidence across scored units | < 0.35 | >= 0.35 |
| `avg_sufficiency` | Is there enough evidence to judge? | >= 0.70 | < 0.70 |
| `low_sufficiency` | Count of scored units below 0.70 sufficiency | 0 | 1+ |
| `high_contradiction` | Count of scored units at/above 0.35 contradiction | 0 | 1+ |

**How the stored results actually read:**
- **Forecast** — both modes abstain in the same way: `escalation_candidate`, zero support, zero contradiction, zero sufficiency.
- **AGI** — both modes escalate, but the committee is less binary than the single juror: contradiction drops from 1.0 to 0.747 and sufficiency rises from 0.0 to 0.23.
- **Control (diversification)** — exact convergence: both modes are `round0_decisive` with support 1.0, contradiction 0.0, sufficiency 1.0.
- **Mixed-truth / High-stakes medical** — both modes flag contradiction strongly, while the committee softens sufficiency relative to the single juror.
- **Legal / AI risk** — both modes converge on a severe reading: contradiction 1.0 and sufficiency 0.0, so both escalate.

In [ ]:
METRICS = [
    "raw_subclaims",
    "aggregated_units",
    "reason",
    "avg_support",
    "avg_contradiction",
    "avg_sufficiency",
    "low_sufficiency",
    "high_contradiction",
]

for r in results:
    s, c = r["single"], r["committee"]
    print(f"\n{'─'*56}")
    print(f"  {r['case']}")
    print(f"  {'':22} {'single':>12} {'committee':>12}")
    print(f"  {'─'*48}")
    for key in METRICS:
        print(f"  {key:22} {str(s.get(key, '-')):>12} {str(c.get(key, '-')):>12}")


────────────────────────────────────────────────────────
  Forecast: rate cut within two quarters
                               single    committee
  ────────────────────────────────────────────────


KeyError: 'scored_units'

#### _In this stored run, the committee is most informative when it softens a binary single-juror read rather than when it fully overturns it._

## What to look for

Both modes use the same model (`gpt-5.4-mini`), so differences come from committee architecture — four personality-injected personas vs one generic call. Each juror's system prompt includes its `Profile` fields (stance, evidence demand, safety bias), so any deltas here are due to reviewer composition rather than model class.

| Pattern | Where to expect it |
|---|---|
| Single and committee both abstain | Forecast under-evidence case |
| Committee gives a softer, less binary read | AGI claim and some medical overclaims |
| Single and committee converge cleanly | Control claim, plus the strongest legal / AI overclaims |

Each `run_case` returns a plain dict — drill into `"single"` / `"committee"` keys to compare any metric. If you rerun the live cells, treat the text above as guidance for this saved run rather than a guarantee of identical future numbers.